<a href="https://colab.research.google.com/github/Balachandar-Ganesan/DeepLearning/blob/main/FeedbackSummarization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from transformers import pipeline, logging
logging.set_verbosity_error()
import pandas as pd

In [2]:
# AI Assistant Class
class Assistant:

  # Object constructor
  def __init__(self):
    pass

  # Function for Sentiment Analysis: Understand the 'vibe' of a text
  def sentiment_analysis(self, review):
    sentiment_app = pipeline("sentiment-analysis",
                             model="distilbert-base-uncased-finetuned-sst-2-english")
    return sentiment_app(review)

  # Function for Summarization: Turn a long paragraph into a bite-sized nugget
  def summarization(self, long_text):
    summarizer = pipeline("summarization",
                          model= "sshleifer/distilbart-cnn-12-6")
    summary = summarizer(long_text, max_length=100, min_length=10)
    return summary[0]['summary_text']

  # Function for Named Entity Recognition (NER): Spot the 'Who', 'Where', and 'What'
  def ner(self, text):
    ner_tagger = pipeline("ner", aggregation_strategy="simple",
                          model="dbmdz/bert-large-cased-finetuned-conll03-english")
    entities = ner_tagger(text)
    entities_dict = [(entity['word'],entity['entity_group']) for entity in entities]
    return entities_dict

  # Function to add a topic to the summary
  def add_topic(self, text):
    topic_classifier = pipeline("zero-shot-classification",
                                # model="MoritzLaurer/mDeBERTa-v3-base-mnli-xnli")
                                model= "facebook/bart-large-mnli")
    topic = topic_classifier(text,
                             candidate_labels=["compliment", "complaint", "suggestion", "statement"])
    return topic['labels'][0], topic['scores'][0]

  # Function to organize the results in a pandas dataframe
  def organize(self, texts):

    # Create an empty dataframe
    df = pd.DataFrame(columns=str.split('customer_name,sentiment,sentiment_conf,topic,topic_conf,summary',
                                        sep=","))
    # Loop through texts
    for text in texts:
      summary = ai.summarization(text)
      topic, topic_conf = ai.add_topic(text)
      sentiment = ai.sentiment_analysis(text)[0]['label']
      sentiment_conf = ai.sentiment_analysis(text)[0]['score']
      customer_name = ai.ner(text)[0][0]
    # Add text to the df
      df.loc[len(df)] = [customer_name, sentiment, sentiment_conf, topic, topic_conf, summary]
    return df

In [3]:
# Instantiate the class
ai = Assistant()

# Run sentiment analysis
review = "I tried the new software update. It's surprisingly intuitive and fast!"
print(f"Sentiment: {ai.sentiment_analysis(review)}")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Sentiment: [{'label': 'POSITIVE', 'score': 0.9995124340057373}]


In [8]:
# Run NER
business_news = "Apple is looking at buying a startup in London for $1 billion."
print(f"Named Entities: {ai.ner(business_news)}")

config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/60.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

Named Entities: [('Apple', 'ORG'), ('London', 'LOC')]


In [9]:
# Run Topic Classifier (Zero-shot classification)
email = """Subject: New Feature Request
Hey, I love your app! One thing that would make it even better is if you could add a dark mode. It would be super helpful for my eyes at night. I've been using it a lot lately and it's really straining my eyes. I've tried changing my phone settings, but it would be great if the app had its own dark mode. That way, I could use it anytime without having to worry about the brightness. Keep up the good work, and I hope you consider my suggestion!
Best, Hanna Lotus."""

topic, score = ai.add_topic(email)
print(f"Topic: {topic}, Score: {score}")

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/1.63G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Topic: suggestion, Score: 0.782721757888794


In [7]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

class TextSummarizer:
    def __init__(self, model_name="sshleifer/distilbart-cnn-12-6"):
        """Initialize the summarizer with a pre-trained model.

        Args:
            model_name (str): Name of the pre-trained model to use.
        """
        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        self.tokenizer = AutoTokenizer.from_pretrained(model_name)
        self.model = AutoModelForSeq2SeqLM.from_pretrained(model_name)
        self.model.to(self.device)

    def summarize(self, text, max_length=130, min_length=30, length_penalty=2.0,
                  repetition_penalty=2.0, num_beams=4, early_stopping=True):
        """Generate a summary for the given text.

        Args:
            text (str): The text to summarize
            max_length (int): Maximum length of the summary
            min_length (int): Minimum length of the summary
            length_penalty (float): Penalty for longer summaries
            repetition_penalty (float): Penalty for repeated tokens
            num_beams (int): Number of beams for beam search
            early_stopping (bool): Whether to stop when all beams are finished

        Returns:
            str: The generated summary
        """
        try:
            # Tokenize the input text
            inputs = self.tokenizer(text, max_length=1024, truncation=True,
                                    padding="max_length", return_tensors="pt"
                                    ).to(self.device)
            # Generate summary
            summary_ids = self.model.generate(
                inputs["input_ids"],
                attention_mask=inputs["attention_mask"],
                max_length=max_length,
                min_length=min_length,
                length_penalty=length_penalty,
                repetition_penalty=repetition_penalty,
                no_repeat_ngram_size=3,
                num_beams=num_beams,
                early_stopping=early_stopping
            )
            # Decode and return the summary
            summary = self.tokenizer.decode(summary_ids[0], skip_special_tokens=True)
            return summary
        except Exception as e:
            print(f"Error during summarization: {str(e)}")
            return text

# Initialize the basic summarizer
summarizer = TextSummarizer()

# Sample text to summarize
sample_text = """
Artificial intelligence (AI) is a rapidly advancing field that focuses on creating intelligent systems
capable of performing tasks that typically require human intelligence. These tasks include natural language
processing, computer vision, speech recognition, and decision-making. With applications across healthcare, finance,
education, and more, AI is transforming the way we interact with technology and solve complex problems.
"""

# Generate a summary
summary = summarizer.summarize(sample_text)
print("Basic Summary:\n", summary)

# Test with shorter text
short_text = "AI is changing the world by automating tasks and providing insights from large datasets."
short_summary = summarizer.summarize(short_text)
print("Short Text Summary:\n", short_summary)

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.22G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/358 [00:00<?, ?it/s]

Basic Summary:
  Artificial intelligence (AI) is transforming the way we interact with technology and solve complex problems . The field focuses on creating intelligent systems capable of performing tasks that typically require human intelligence . These tasks include natural language processing, computer vision, speech recognition and decision-making .
Short Text Summary:
  AI is changing the world by automating tasks and providing insights from large datasets . It's changing the way we look at our jobs in a new way to make sure we don't have to rely on relying on humans .
